# 04 — Đánh giá nội bộ open-set

Chạy sau `03_train_model.ipynb`. Notebook này đánh giá time split, thiết bị chưa thấy và lớp chưa thấy trên tập CIC đã xác minh; không dùng IoT Sentinel để chọn ngưỡng. Định nghĩa model được nạp từ `03` mà không train lại.


In [5]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

Configured for 88-core CPU


In [6]:
# Nạp định nghĩa từ 03 mà không chạy train.
from pathlib import Path

_cwd = Path.cwd().resolve()
_ROOT = next((p for p in (_cwd, *_cwd.parents)
              if (p / "Code" / "03_train_model.ipynb").is_file()), None)
assert _ROOT is not None, f"Không thấy Code/03_train_model.ipynb quanh {_cwd}"
_CODE = _ROOT / "Code"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_CODE / "03_train_model.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import pandas as pd
from IPython.display import display

import json


## 1. Chọn candidate

Điền `EVALUATE_RUN` khi có nhiều candidate. Nếu để trống, ưu tiên run đã ghim trong `current_model.json`; khi chưa ghim, chỉ tự chọn nếu có đúng một candidate.


In [7]:
EVALUATE_RUN = "20260915_132703_verified_tiered"   # ví dụ: "20260913_162157_verified_tiered"
RECOMPUTE_UNSEEN_DEVICE = False
RECOMPUTE_UNSEEN_CLASS = False

run_dir = select_run_dir(EVALUATE_RUN or globals().get("run_dir"))
print("Đánh giá:", run_dir)


Đánh giá: /home/ubuntu/sepcung/02.SDC/Models/20260915_132703_verified_tiered


## 2. Chạy đánh giá

Có thể tái dùng OOF trong candidate cho hai chế độ holdout. Đặt `RECOMPUTE_* = True` khi cần tính lại từ dataset hiện tại.


In [8]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module=r"sklearn\.utils\.parallel")

evaluation_args = ["--run", str(run_dir)]
if RECOMPUTE_UNSEEN_DEVICE:
    evaluation_args.append("--recompute-unseen-device")
if RECOMPUTE_UNSEEN_CLASS:
    evaluation_args.append("--recompute-unseen-class")
internal_report_dir = evaluate_main(evaluation_args).resolve()
metrics = pd.read_csv(internal_report_dir / "metrics.csv")
headline = metrics[metrics["breakdown"].eq("all") & metrics["level"].isin(["window", "device"])]
display(headline.reset_index(drop=True))


run=20260915_132703_verified_tiered; sessions=8189; features=40
reuse unseen_class from /home/ubuntu/sepcung/02.SDC/Models/20260915_132703_verified_tiered/oof_pred.parquet
wrote /home/ubuntu/sepcung/02.SDC/Reports/open_set_20260915_132703_verified_tiered


,regime,head,population,level,breakdown,value,n,n_answered,coverage,accuracy_answered,false_positive_rate
0,time,make,all,window,all,all,1650,1206,0.730909,1.000000,NaN
1,time,make,known,window,all,all,1600,1206,0.753750,1.000000,NaN
2,time,make,unknown,window,all,all,50,0,0.000000,NaN,0.000000
3,time,model,all,window,all,all,1650,1167,0.707273,1.000000,NaN
4,time,model,known,window,all,all,1600,1167,0.729375,1.000000,NaN
5,time,model,unknown,window,all,all,50,0,0.000000,NaN,0.000000
6,time,type,all,window,all,all,1650,1170,0.709091,1.000000,NaN
7,time,type,known,window,all,all,1606,1170,0.728518,1.000000,NaN
8,time,type,unknown,window,all,all,44,0,0.000000,NaN,0.000000
9,unseen_class,make,all,window,all,all,8189,10,0.001221,0.000000,NaN


## 3. Kiểm tra báo cáo

Báo cáo và dự đoán chi tiết nằm trong `Reports/open_set_<run_id>/`. `05_model_out.ipynb` kiểm tra báo cáo này trước khi kích hoạt candidate.


In [9]:
report_summary = json.loads((internal_report_dir / "summary.json").read_text(encoding="utf-8"))
assert report_summary["run_id"] == run_dir.name
assert report_summary["external_test_used_for_tuning"] is False
print("Báo cáo:", internal_report_dir)
print("Xung đột hierarchy:", report_summary["hierarchy_conflicts"])
display(pd.read_csv(internal_report_dir / "calibration.csv"))


Báo cáo: /home/ubuntu/sepcung/02.SDC/Reports/open_set_20260915_132703_verified_tiered
Xung đột hierarchy: 0


,regime,head,bin,n,mean_confidence,empirical_accuracy
0,time,make,"(0.3, 0.4]",2,0.360000,0.000000
1,time,make,"(0.6, 0.7]",1,0.680000,1.000000
2,time,make,"(0.7, 0.8]",6,0.773333,1.000000
3,time,make,"(0.9, 1.0]",1096,0.999124,1.000000
4,time,model,"(0.5, 0.6]",2,0.540000,0.000000
5,time,model,"(0.6, 0.7]",4,0.680000,1.000000
6,time,model,"(0.7, 0.8]",12,0.760000,1.000000
7,time,model,"(0.8, 0.9]",35,0.872000,1.000000
8,time,model,"(0.9, 1.0]",1052,0.999125,1.000000
9,time,type,"(0.4, 0.5]",6,0.480000,1.000000
